# Soil Quality Classification Using Random Forest and Decision Tree

**Roll Numbers:** 2520030518, 2520030524, 2520030534
**Department:** Computer Science Engineering, KLH University
**Course:** Machine Learning (25SC2107E) | **Guide:** Mr. G. Ravindra Babu

This notebook builds a machine learning system that classifies **soil quality**
from soil parameters (pH, nitrogen, phosphorus, potassium, organic matter and moisture)
using two supervised tree-based algorithms: a **Decision Tree** and a **Random Forest**.
Following the project abstract, we cover: preprocessing and missing-value imputation,
feature engineering, Gini vs. entropy splitting, pruning, bagging with feature subsampling,
cross-validation, evaluation (accuracy, precision, recall, F1-score, confusion matrix)
and feature-importance analysis.

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, GridSearchCV)
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (7, 5)
RANDOM_STATE = 42

## 2. Load the Dataset

**Set these two values for your dataset:** the CSV file name and the name of the target column
(the soil quality label). The notebook then adapts to whatever feature columns the file contains.

> **Important:** if `soil_quality.csv` is not found, the notebook generates a small
> **synthetic demo dataset** so that every cell can run. Results on the demo data are **not real** —
> place your real CSV next to this notebook (or upload it in Colab) and re-run before using any numbers in your report.

In [ ]:
import os

CSV_PATH   = 'soil_quality.csv'     # <-- your dataset file
TARGET_COL = 'soil_quality'         # <-- name of the soil-quality label column

USING_SYNTHETIC = False
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    USING_SYNTHETIC = True
    print("WARNING: '%s' not found -> using SYNTHETIC DEMO DATA (results are NOT real)." % CSV_PATH)
    rng = np.random.default_rng(RANDOM_STATE)
    n = 1500
    df = pd.DataFrame({
        'ph':             rng.normal(6.6, 0.9, n).clip(4, 9),
        'nitrogen':       rng.normal(280, 70, n).clip(50, 600),
        'phosphorus':     rng.normal(30, 10, n).clip(2, 80),
        'potassium':      rng.normal(190, 50, n).clip(40, 400),
        'organic_matter': rng.normal(2.6, 0.9, n).clip(0.2, 6),
        'moisture':       rng.normal(24, 7, n).clip(5, 50),
    })
    score = (-abs(df.ph - 6.8) * 1.2 + df.nitrogen / 120 + df.phosphorus / 20
             + df.potassium / 100 + df.organic_matter * 0.9 - abs(df.moisture - 25) / 8
             + rng.normal(0, 0.7, n))
    df[TARGET_COL] = pd.qcut(score, [0, .3, .7, 1.0], labels=['Poor', 'Moderate', 'Good']).astype(str)
    for col in ['ph', 'nitrogen', 'potassium', 'moisture']:      # inject some missing values
        df.loc[rng.choice(n, int(0.04 * n), replace=False), col] = np.nan

print("Dataset shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe().T

## 3. Exploratory Data Analysis (EDA)

Before modeling, we look at the soil-quality class balance and how each soil parameter relates to quality.

In [ ]:
plt.figure()
order = df[TARGET_COL].value_counts().index
sns.countplot(x=TARGET_COL, data=df, order=order, palette='viridis')
plt.title('Soil Quality Class Distribution')
plt.ylabel('Number of Samples')
plt.show()

print(df[TARGET_COL].value_counts(normalize=True).round(3) * 100)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
n = len(num_cols)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(4.2 * ((n + 1) // 2), 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(data=df, x=col, hue=TARGET_COL, kde=True, ax=ax, element='step')
    ax.set_title(f'{col} by soil quality')
for ax in axes.ravel()[n:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(4.2 * ((n + 1) // 2), 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(data=df, x=TARGET_COL, y=col, ax=ax, palette='viridis')
    ax.set_title(f'{col} vs soil quality')
for ax in axes.ravel()[n:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap of Soil Parameters')
plt.show()

## 4. Data Preprocessing and Feature Engineering

Steps:
1. Drop duplicate rows and any identifier columns (e.g. `id`, `sample_id`) — they are not predictive.
2. Encode the target label (e.g. Poor / Moderate / Good) as integers.
3. One-hot encode any categorical feature columns (if the dataset has any).
4. **Feature engineering:** add a few simple soil-nutrient features (total NPK and nutrient ratios) when N, P, K columns exist.
5. Stratified train/test split (80/20) so both sets keep the same class ratio.
6. **Missing values** are imputed with the training-set median *inside a Pipeline*, so the imputer is fitted on training data only (no data leakage).
7. Feature scaling is **not required** for tree-based models, so it is skipped.

In [ ]:
data = df.drop_duplicates().copy()

# drop identifier-like columns if present
id_cols = [c for c in data.columns if c.lower() in ('id', 'sample_id', 'sampleid', 'index', 'unnamed: 0')]
data = data.drop(columns=id_cols)

# drop rows where the target itself is missing
data = data.dropna(subset=[TARGET_COL])

# ---- target encoding (keep a sensible class order if the labels are Poor/Moderate/Good)
target_vals = data[TARGET_COL].astype(str)
preferred = ['Poor', 'Moderate', 'Good']
if set(target_vals.unique()) == set(preferred):
    class_names = preferred
else:
    class_names = sorted(target_vals.unique())
label_map = {name: i for i, name in enumerate(class_names)}
y = target_vals.map(label_map)

# ---- features
X = data.drop(columns=[TARGET_COL])
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
if cat_cols:
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# ---- simple feature engineering (row-wise, so no leakage)
lower = {c.lower(): c for c in X.columns}
n_col = lower.get('nitrogen', lower.get('n'))
p_col = lower.get('phosphorus', lower.get('p'))
k_col = lower.get('potassium', lower.get('k'))
if n_col and p_col and k_col:
    X['npk_total'] = X[n_col] + X[p_col] + X[k_col]
    X['n_to_p_ratio'] = X[n_col] / (X[p_col] + 1e-6)
    X['n_to_k_ratio'] = X[n_col] / (X[k_col] + 1e-6)

print("Classes:", label_map)
print("Final feature columns:", list(X.columns))
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", X_train.shape, " Test size:", X_test.shape)
print("Missing values in training features:", int(X_train.isnull().sum().sum()))

def make_pipeline(model):
    # median imputation is fitted on training data only
    return Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', model)])

## 5. Model 1 — Decision Tree

A Decision Tree repeatedly splits the data on the soil parameter that best separates the classes.
We first compare the two splitting criteria from the abstract — **Gini impurity** and **entropy** —
and then show how an unrestricted tree **overfits**.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for crit in ['gini', 'entropy']:
    pipe = make_pipeline(DecisionTreeClassifier(criterion=crit, random_state=RANDOM_STATE))
    scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='accuracy')
    rows.append({'Criterion': crit, 'CV Accuracy (mean)': scores.mean(), 'CV Std': scores.std()})
pd.DataFrame(rows).set_index('Criterion').round(4)

### Overfitting and the bias–variance trade-off
Deeper trees fit the training data better, but past a certain depth the test accuracy stops improving or drops.

In [ ]:
depths = range(1, 21)
train_acc, test_acc = [], []
imp = SimpleImputer(strategy='median').fit(X_train)
Xtr_i, Xte_i = imp.transform(X_train), imp.transform(X_test)
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(Xtr_i, y_train)
    train_acc.append(t.score(Xtr_i, y_train)); test_acc.append(t.score(Xte_i, y_test))

plt.plot(depths, train_acc, 'o-', label='Train accuracy')
plt.plot(depths, test_acc, 's-', label='Test accuracy')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Decision Tree: Depth vs Accuracy (overfitting)')
plt.legend(); plt.show()

### Pruning (cost-complexity pruning)
Pruning removes branches that add little predictive value. We use `ccp_alpha`:
larger values prune more. The best alpha is chosen by **cross-validation on the training set only**.

In [ ]:
path = DecisionTreeClassifier(random_state=RANDOM_STATE).cost_complexity_pruning_path(Xtr_i, y_train)
alphas = np.unique(path.ccp_alphas.clip(min=0))
# keep a manageable number of candidate alphas
alphas = alphas[:: max(1, len(alphas) // 25)]

cv_means = []
for a in alphas:
    pipe = make_pipeline(DecisionTreeClassifier(ccp_alpha=a, random_state=RANDOM_STATE))
    cv_means.append(cross_val_score(pipe, X_train, y_train, cv=skf, scoring='accuracy').mean())

best_alpha = alphas[int(np.argmax(cv_means))]
plt.plot(alphas, cv_means, 'o-')
plt.axvline(best_alpha, color='r', ls='--', label=f'best ccp_alpha = {best_alpha:.5f}')
plt.xlabel('ccp_alpha'); plt.ylabel('CV accuracy')
plt.title('Pruning: choosing ccp_alpha by cross-validation')
plt.legend(); plt.show()
print("Best ccp_alpha:", best_alpha)

In [ ]:
# Final Decision Tree models: unpruned baseline and pruned version
dt_full = make_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)).fit(X_train, y_train)
dt = make_pipeline(DecisionTreeClassifier(criterion='gini', ccp_alpha=best_alpha,
                                          random_state=RANDOM_STATE)).fit(X_train, y_train)

print("Unpruned tree depth / leaves:", dt_full.named_steps['model'].get_depth(),
      "/", dt_full.named_steps['model'].get_n_leaves())
print("Pruned   tree depth / leaves:", dt.named_steps['model'].get_depth(),
      "/", dt.named_steps['model'].get_n_leaves())

dt_pred = dt.predict(X_test)
dt_full_pred = dt_full.predict(X_test)

In [ ]:
plt.figure(figsize=(16, 7))
plot_tree(dt.named_steps['model'], feature_names=list(X.columns), class_names=class_names,
          filled=True, rounded=True, max_depth=3, fontsize=8)
plt.title('Pruned Decision Tree (top 3 levels shown)')
plt.show()

## 6. Model 2 — Random Forest

Random Forest builds many decision trees, each trained on a **bootstrap sample** of the data (**bagging**)
and using a **random subset of features** at every split (**feature subsampling**). Averaging their votes
reduces variance and improves generalization to unseen soil samples.

In [ ]:
rf = make_pipeline(RandomForestClassifier(
    n_estimators=200, max_features='sqrt', bootstrap=True, oob_score=True,
    random_state=RANDOM_STATE, n_jobs=-1
)).fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)
print("Out-of-bag (OOB) accuracy estimate:", round(rf.named_steps['model'].oob_score_, 4))

## 7. Cross-Validation and Hyperparameter Tuning

Cross-validation gives a more reliable performance estimate than a single split and helps us check
that the models are stable. We use **5-fold stratified cross-validation** on the training set.

In [ ]:
cv_results = {}
models_cv = {'Decision Tree (unpruned)': dt_full, 'Decision Tree (pruned)': dt, 'Random Forest': rf}
for name, m in models_cv.items():
    cv_results[name] = cross_val_score(m, X_train, y_train, cv=skf, scoring='accuracy')

cv_df = pd.DataFrame({k: [v.mean(), v.std()] for k, v in cv_results.items()},
                     index=['CV Mean Accuracy', 'CV Std']).T
display(cv_df.round(4))

plt.boxplot(cv_results.values(), labels=cv_results.keys())
plt.ylabel('CV accuracy'); plt.title('5-Fold Cross-Validation Accuracy')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

### Hyperparameter tuning (GridSearchCV)
Tuning uses cross-validation on the **training data only**; the test set stays untouched until the final evaluation.
(The grids are small so that the notebook runs quickly — enlarge them for a deeper search.)

In [ ]:
dt_grid = GridSearchCV(
    make_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
    {'model__criterion': ['gini', 'entropy'],
     'model__max_depth': [3, 5, 7, 10, None],
     'model__min_samples_leaf': [1, 5, 10],
     'model__ccp_alpha': [0.0, best_alpha]},
    cv=skf, scoring='accuracy', n_jobs=-1).fit(X_train, y_train)

rf_grid = GridSearchCV(
    make_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
    {'model__n_estimators': [100, 200, 300],
     'model__max_depth': [None, 10, 20],
     'model__max_features': ['sqrt', 'log2'],
     'model__min_samples_leaf': [1, 3]},
    cv=skf, scoring='accuracy', n_jobs=-1).fit(X_train, y_train)

print("Best Decision Tree params:", dt_grid.best_params_, "| CV acc:", round(dt_grid.best_score_, 4))
print("Best Random Forest params:", rf_grid.best_params_, "| CV acc:", round(rf_grid.best_score_, 4))

dt_tuned, rf_tuned = dt_grid.best_estimator_, rf_grid.best_estimator_
dt_tuned_pred, rf_tuned_pred = dt_tuned.predict(X_test), rf_tuned.predict(X_test)

## 8. Model Evaluation

All models are evaluated on the **same held-out test set** using Accuracy, Precision, Recall and F1-score
(macro-averaged, so every soil-quality class counts equally), plus the confusion matrix.

In [ ]:
def get_metrics(name, y_true, y_pred):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision (macro)': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall (macro)': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'F1-score (macro)': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

results = pd.DataFrame([
    get_metrics('Decision Tree (unpruned)', y_test, dt_full_pred),
    get_metrics('Decision Tree (pruned)',   y_test, dt_pred),
    get_metrics('Decision Tree (tuned)',    y_test, dt_tuned_pred),
    get_metrics('Random Forest',            y_test, rf_pred),
    get_metrics('Random Forest (tuned)',    y_test, rf_tuned_pred),
]).set_index('Model')
results.round(4)

In [ ]:
results.round(3).plot(kind='bar', figsize=(11, 6), colormap='viridis')
plt.title('Decision Tree vs Random Forest — Metric Comparison')
plt.ylabel('Score'); plt.ylim(0, 1.05)
plt.xticks(rotation=15); plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

### Confusion Matrices

In [ ]:
to_plot = [('Decision Tree (pruned)', dt_pred), ('Random Forest', rf_pred)]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, pred) in zip(axes, to_plot):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(f'{name} — Confusion Matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

### Detailed Classification Reports

In [ ]:
print("Decision Tree (pruned):\n", classification_report(y_test, dt_pred, target_names=class_names))
print("Random Forest:\n", classification_report(y_test, rf_pred, target_names=class_names))

### Optional: ROC-AUC (one-vs-rest)

In [ ]:
try:
    print("Decision Tree (pruned) ROC-AUC:", round(roc_auc_score(y_test, dt.predict_proba(X_test), multi_class='ovr'), 4))
    print("Random Forest        ROC-AUC:", round(roc_auc_score(y_test, rf_proba, multi_class='ovr'), 4))
except Exception as e:
    print("ROC-AUC not computed:", e)

## 9. Feature Importance

Which soil attributes matter most for predicting soil quality? This gives farmers and agronomists
*interpretable* insight into what drives the classification, not just the prediction.

In [ ]:
dt_imp = pd.Series(dt.named_steps['model'].feature_importances_, index=X.columns).sort_values(ascending=False)
rf_imp = pd.Series(rf.named_steps['model'].feature_importances_, index=X.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
dt_imp.plot(kind='barh', ax=axes[0], color='#2E86AB'); axes[0].invert_yaxis()
axes[0].set_title('Decision Tree (pruned) — Feature Importance')
rf_imp.plot(kind='barh', ax=axes[1], color='#2A9D8F'); axes[1].invert_yaxis()
axes[1].set_title('Random Forest — Feature Importance')
plt.tight_layout(); plt.show()

pd.DataFrame({'Decision Tree': dt_imp, 'Random Forest': rf_imp}).round(4)

## 10. Predicting the Quality of New Soil Samples

Once trained, the model gives an instant prediction (and a confidence for each class) for any new soil sample,
which is the practical value of the project: no waiting for lab analysis.

In [ ]:
# Take a few unseen test samples and show predictions with class probabilities
sample = X_test.head(8).copy()
pred_ids = rf.predict(sample)
proba = rf.predict_proba(sample)

out = sample.copy()
out['Actual'] = [class_names[i] for i in y_test.head(8)]
out['Predicted'] = [class_names[i] for i in pred_ids]
for i, c in enumerate(class_names):
    out[f'P({c})'] = proba[:, i].round(3)
out

In [ ]:
def predict_soil_quality(model, **soil_values):
    """Predict soil quality for one new sample. Pass the ORIGINAL columns by name,
    e.g. predict_soil_quality(rf, ph=6.5, nitrogen=280, phosphorus=30, potassium=190,
                                  organic_matter=2.6, moisture=24)"""
    row = pd.DataFrame([soil_values])
    # rebuild the same engineered features used in training
    lower_ = {c.lower(): c for c in row.columns}
    n_, p_, k_ = lower_.get('nitrogen', lower_.get('n')), lower_.get('phosphorus', lower_.get('p')), lower_.get('potassium', lower_.get('k'))
    if n_ and p_ and k_:
        row['npk_total'] = row[n_] + row[p_] + row[k_]
        row['n_to_p_ratio'] = row[n_] / (row[p_] + 1e-6)
        row['n_to_k_ratio'] = row[n_] / (row[k_] + 1e-6)
    row = row.reindex(columns=X.columns, fill_value=0)
    probs = model.predict_proba(row)[0]
    return class_names[int(np.argmax(probs))], dict(zip(class_names, probs.round(3)))

# Example call (edit column names/values to match your dataset)
# predict_soil_quality(rf, ph=6.5, nitrogen=280, phosphorus=30, potassium=190, organic_matter=2.6, moisture=24)

## 11. Conclusion

> Fill in the numbers below from **your real dataset run** (not the synthetic demo data).

- A **Decision Tree** and a **Random Forest** were trained on `[N]` soil samples using `[list features]`
  to classify soil quality into `[classes]`.
- Missing values were imputed with the median (fitted on training data only), features were encoded/engineered,
  and a stratified 80/20 split plus 5-fold cross-validation were used to prevent data leakage.
- Gini and entropy criteria were compared; **pruning** reduced tree size from `[x]` to `[y]` leaves and improved
  generalization compared with the unpruned tree.
- On the held-out test set, the best model was **`[model]`** with accuracy `[x]`, macro-F1 `[x]`
  (see the comparison table in Section 8). Random Forest `[did / did not]` outperform the single Decision Tree
  in accuracy and stability (CV standard deviation `[x]` vs `[y]`).
- The most influential soil attributes were `[feature 1]`, `[feature 2]` and `[feature 3]`.
- Such a model can give farmers fast, explainable soil-quality predictions to support crop planning
  and fertilizer decisions.

**Possible extensions:** larger multi-region datasets, Gradient Boosting / XGBoost, SHAP explanations,
IoT sensor integration, and a Flask/Streamlit app for farmers.